# Day 5: Silver Standardization & Entity Linkage

This notebook maps events to OMOP CDM v5.4 and uses Splink for probabilistic patient linkage.

In [ ]:
import splink.spark.comparison_library as cl
from splink.spark.linker import SparkLinker
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Day05_Splink_OMOP") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()


## 1. OMOP Mock Transformation

In [ ]:
from pyspark.sql.functions import col, lit, monotonically_increasing_id

events_data = [
    ("pat_1", "diagnosis", "2023-01-01", "J45.909"),
    ("pat_1", "medication", "2023-01-02", "RX123"),
    ("pat_2", "diagnosis", "2023-02-15", "E11.9")
]
events_df = spark.createDataFrame(events_data, ["patient_id", "event_type", "event_date", "code"])

condition_occurrence = events_df.filter(col("event_type") == "diagnosis") \
    .withColumn("condition_occurrence_id", monotonically_increasing_id()) \
    .withColumn("person_id", col("patient_id")) \
    .withColumn("condition_concept_id", lit(0)) \
    .withColumn("condition_start_date", col("event_date")) \
    .withColumn("condition_source_value", col("code")) \
    .select("condition_occurrence_id", "person_id", "condition_concept_id", "condition_start_date", "condition_source_value")

print("OMOP Condition Occurrence:")
condition_occurrence.show()


## 2. Splink Entity Linkage

In [ ]:
patient_data = [
    {"unique_id": "1", "first_name": "John", "last_name": "Doe", "dob": "1980-01-01", "city": "New York"},
    {"unique_id": "2", "first_name": "Jon", "last_name": "Doe", "dob": "1980-01-01", "city": "New York"},
    {"unique_id": "3", "first_name": "Jane", "last_name": "Smith", "dob": "1990-05-15", "city": "Boston"}
]
df_patients = spark.createDataFrame(patient_data)

settings = {
    "link_type": "dedupe_only",
    "blocking_rules_to_generate_predictions": [
        "l.dob = r.dob",
        "l.city = r.city"
    ],
    "comparisons": [
        cl.exact_match("first_name"),
        cl.exact_match("last_name"),
        cl.exact_match("dob")
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = SparkLinker(df_patients, settings, spark=spark)
linker.estimate_u_using_random_sampling(max_pairs=1000)
linker.estimate_parameters_using_expectation_maximisation("l.dob = r.dob")

df_predictions = linker.predict(threshold_match_probability=0.5)
print("Splink Linkage Predictions:")
df_predictions.as_pandas_dataframe()
